# Basic Pipeline: SED Modelling of MG3 J163554+3629

**Author:** [Your Name]  
**Object:** MG3 J163554+3629 — High-redshift FSRQ blazar (z = 3.648)  
**Purpose:** This notebook documents the standard building blocks used across all analyses in this repository. It serves as both a tutorial and a reusable template.

---

## Overview of the pipeline

| Step | What it does |
|------|--------------|
| 1 | Load and inspect raw SED data |
| 2 | Add systematics and flag upper limits |
| 3 | Bin the data (optional — compare binned vs unbinned) |
| 4 | SED shape characterisation (sync + IC peak finding) |
| 5 | Physical prefit using ObsConstrain |
| 6 | Add geometry: opening angle + EBL absorption |

---

## Step 0 — Imports

We use **JetSeT** as the core SED modelling framework. It handles data loading, jet model construction, emission calculations (synchrotron, SSC, EC), and interfaces with fitting tools like Minuit and emcee.

In [1]:
import jetset
print('JetSeT version:', jetset.__version__)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from jetset.data_loader import Data, ObsData
from jetset.sed_shaper import SEDShape
from jetset.jet_model import Jet
from jetset.obs_constrain import ObsConstrain
from jetset.model_manager import FitModel
from jetset.template_2Dmodel import EBLAbsorptionTemplate

JetSeT version: 1.3.0


---
## Step 1 — Load the raw SED data

The file `SED.txt` contains three columns: **frequency (Hz)**, **flux νF_ν (erg cm⁻² s⁻¹)**, and **flux error**. A zero in the error column indicates an upper limit.

We create **two independent ObsData objects** from the same raw data:
- `raw_data` — untouched, for plotting and reference
- `sed_data` — this is what we will process (add errors, bin, flag ULs)

We also remove one spurious Fermi point at 1.709×10²² Hz that is physically inconsistent with the rest of the gamma-ray data (likely a bad detection or artefact).

In [2]:
# Load raw data — adjust path if running from a subfolder
d = np.genfromtxt('../../data/SED.txt')

# Remove spurious Fermi point
d = d[d[:, 0] != 1.709E22]

print(f'Total data points loaded: {d.shape[0]}')
print(f'Frequency range: {d[:,0].min():.2e} — {d[:,0].max():.2e} Hz')

# Build JetSeT Data object
data = Data(n_rows=d.shape[0])
data.set_field('x',  d[:, 0])   # frequency in Hz
data.set_field('y',  d[:, 1])   # flux nuFnu in erg/cm^2/s
data.set_field('dy', d[:, 2])   # flux error (0 = upper limit)

# Object metadata
data.set_meta_data('z',          3.647763)   # redshift
data.set_meta_data('UL_CL',      0.95)       # upper limit confidence level
data.set_meta_data('restframe',  'obs')      # data in observer frame
data.set_meta_data('data_scale', 'lin-lin')  # both axes are linear (not log)
data.set_meta_data('obj_name',   'MG3J163554+3629')

# Two independent ObsData objects from the same raw table
raw_data = ObsData(data_table=data)   # reference copy — never modified
sed_data = ObsData(data_table=data)   # working copy — will be processed below

# Plot the raw SED
myPlot = raw_data.plot_sed()
myPlot.setlim(y_min=1E-16, x_min=1E8)
plt.title('MG3 J163554+3629 — Raw SED')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '../../data/SED.txt'

---
## Step 2 — Add systematics and flag upper limits

### Why add systematics?
Multi-wavelength SED data come from many different telescopes (radio, optical, X-ray, gamma-ray), each with their own calibration uncertainties. These **systematic errors** (e.g. flux calibration offsets, cross-instrument discrepancies) are not captured in the statistical error bars. If we fit without accounting for them, the fit is dominated by the instruments with the smallest statistical errors, which is misleading.

We add a **15% systematic floor** across the full frequency range (10⁶ to 10²⁹ Hz). This means any data point whose error is smaller than 15% of its flux value gets its error increased to that floor.

### Why flag upper limits?
Some data points are **non-detections** — the telescope did not detect the source but can place an upper bound on the flux. These must be treated differently in the likelihood: the model is only penalised if it predicts *more* flux than the upper limit, not if it predicts less.

Points with zero error in `SED.txt` are upper limits. We also manually flag specific frequencies that we know to be upper limits from the observational context.

In [ ]:
# Remove any data flagged as bad quality by JetSeT internally
sed_data.filter_data_set('-1', exclude=True)

# Add 15% systematic error across the full SED
# Arguments: (systematic fraction, [freq_min, freq_max])
sed_data.add_systematics(0.15, [10.**6, 10.**29])

# Manually flag known upper limits by frequency
# These are non-detections: gamma-ray (Fermi), X-ray (one point), sub-mm
UL_freq = [
    1.324e25,   # gamma-ray (Fermi)
    7.646e25,   # gamma-ray (Fermi)
    4.188e24,   # gamma-ray (Fermi)
    4.188e22,   # X-ray upper limit
    2.417e13,   # sub-mm (Herschel/SCUBA)
    1.318e13,   # sub-mm
]

for freq in UL_freq:
    # rtol=0.05 allows 5% frequency tolerance when matching
    mask = np.isclose(sed_data.table['nu_data'], freq, rtol=0.05)
    sed_data.table['UL'][mask] = True
    
print(f'Upper limits flagged: {sed_data.table["UL"].sum()}')
print(f'Total points in working dataset: {len(sed_data.table)}')

---
## Step 3 — Binning (optional)

### What is binning?
Binning groups nearby data points in frequency space into a single averaged point. We use `bin_width=0.15` which means points within 0.15 dex (in log frequency) of each other are merged.

### Why bin?
- **Reduces noise**: Dense regions (e.g. optical/UV with many archival measurements) can over-constrain the fit in one frequency range while sparse regions are under-weighted
- **Speeds up fitting**: Fewer points = faster likelihood evaluations, especially for MCMC and nested sampling
- **Avoids correlated data issues**: Multiple observations at the same frequency are not truly independent

### Binned vs unbinned
Both approaches are valid and we carry out analyses with each (see `analysis/one_zone/`). The comparison between them tests whether our results are robust to this choice.

In [ ]:
# --- UNBINNED version (comment out binning line below to use this) ---
sed_data_unbinned = ObsData(data_table=data)
sed_data_unbinned.filter_data_set('-1', exclude=True)
sed_data_unbinned.add_systematics(0.15, [10.**6, 10.**29])
for freq in UL_freq:
    mask = np.isclose(sed_data_unbinned.table['nu_data'], freq, rtol=0.05)
    sed_data_unbinned.table['UL'][mask] = True

# --- BINNED version ---
sed_data_binned = ObsData(data_table=data)
sed_data_binned.filter_data_set('-1', exclude=True)
sed_data_binned.group_data(bin_width=0.15)   # <-- this is the only difference
sed_data_binned.add_systematics(0.15, [10.**6, 10.**29])
for freq in UL_freq:
    mask = np.isclose(sed_data_binned.table['nu_data'], freq, rtol=0.05)
    sed_data_binned.table['UL'][mask] = True

print(f'Unbinned: {len(sed_data_unbinned.table)} points')
print(f'Binned:   {len(sed_data_binned.table)} points')

# Overlay both on same plot to visualise the difference
myPlot = raw_data.plot_sed()
myPlot.add_data_plot(sed_data_binned,   label='binned',   color='red',  fmt='s', ms=5)
myPlot.add_data_plot(sed_data_unbinned, label='unbinned', color='blue', fmt='o', ms=4, alpha=0.5)
myPlot.setlim(y_min=1E-16, x_min=1E8)
plt.title('MG3 J163554+3629 — Binned vs Unbinned')
plt.show()

# For all subsequent steps we use the UNBINNED version as default
# (swap to sed_data_binned for binned analyses)
sed_data = sed_data_unbinned

---
## Step 4 — SED shape characterisation

### What is SEDShape doing?
Before building a physical jet model, JetSeT fits simple **phenomenological functions** to the two broad humps in the SED:
- **Synchrotron peak** (radio → UV): fitted with a log-parabola in the range 10⁹–10¹⁶ Hz
- **Inverse Compton peak** (X-ray → gamma-ray): fitted with a power law in the range 10¹⁶–10²⁶ Hz

### Why do this?
This shape fit gives us the **peak frequencies and luminosities** of both humps, which `ObsConstrain` (Step 5) then uses to derive physically motivated starting values for the jet parameters. Without this step, finding a reasonable starting point for the physical model would require manual guesswork.

The `check_BBB_template=True` option checks whether a Big Blue Bump (accretion disk thermal emission) is visible in the optical/UV — important for FSRQs like this one.

In [ ]:
my_shape = SEDShape(sed_data)
my_shape.eval_indices(silent=True)

# Synchrotron peak fit
# use_log_par=True: fits a log-parabola (more physical than a simple parabola)
# minimizer='lsb': least-squares Levenberg-Marquardt (fast, good for smooth peaks)
my_shape.sync_fit(
    check_BBB_template=True,
    check_host_gal_template=False,  # no host galaxy at z=3.65
    use_log_par=True,
    minimizer='lsb',
    fit_range=[9, 16],              # log10(nu) range for sync peak
    silent=True
)

# Inverse Compton peak fit
# minimizer='minuit': more robust for the IC hump which can be irregular
my_shape.IC_fit(
    fit_range=[16, 26],             # log10(nu) range for IC peak
    minimizer='minuit',
    silent=True
)

# Plot the shape fit
p = my_shape.plot_shape_fit()
p.setlim(y_min=1E-16)
plt.title('SED shape characterisation — sync + IC peak fits')
plt.show()

print('>>> SED shape characterisation complete.')
print(f'    Sync peak frequency  : {10**my_shape.table["nu_p_S"][0]:.2e} Hz')
print(f'    IC peak frequency    : {10**my_shape.table["nu_p_IC"][0]:.2e} Hz')

---
## Step 5 — Physical prefit with ObsConstrain

### What is ObsConstrain doing?
`ObsConstrain` uses the SED shape results (peak frequencies, luminosities, variability timescale) to derive **physically self-consistent starting values** for the jet model parameters via analytical equipartition arguments and synchrotron cooling theory.

### Physical model: SSC + EC
For MG3 J163554+3629 — a high-redshift FSRQ — the IC hump is dominated by **External Compton (EC)** processes: electrons in the jet upscatter photons from external radiation fields:
- **EC_DT**: scattering off the Dust Torus (warm IR photons)
- **EC_BLR**: scattering off the Broad Line Region (UV/optical photons)
- **EC_Disk**: scattering off the accretion disk directly

### Key fixed parameters and why
- `theta=3°`: jet viewing angle — constrained by radio VLBI morphology
- `bulk_factor=18.5`: initial Lorentz factor estimate from superluminal motion / SED shape
- `B_range=[0.02, 0.08]` G: magnetic field range physically reasonable for FSRQs at this distance from the black hole
- `t_var_sec=7*86400`: 7-day variability timescale — constrains the emission region size via R ≤ c × t_var × δ
- `distr_e='bkn'`: broken power law electron energy distribution — needed to reproduce both the spectral slope below and above the synchrotron peak

### Central engine parameters
The ObsConstrain guess for M_BH, L_Disk etc. is rough. We override these with values refined from the Sánchez Zaballa (2025) paper and our own Minuit fits.

In [ ]:
sed_obspar = ObsConstrain(
    theta=3,                      # viewing angle in degrees
    bulk_factor=18.5,             # bulk Lorentz factor
    B_range=[0.02, 0.08],        # magnetic field range (Gauss)
    distr_e='bkn',               # broken power law electron distribution
    t_var_sec=7*86400,           # variability timescale: 7 days in seconds
    nu_cut_IR=1E9,               # low-frequency cutoff for IR template
    SEDShape=my_shape            # pass the shape fit results
)

# Build the physical prefit jet
# EC components: Dust Torus + Broad Line Region + Accretion Disk
# disk_type='MultiBB': multi-temperature blackbody disk (more physical than single BB)
prefit_jet = sed_obspar.constrain_SSC_EC_model(
    electron_distribution_log_values=False,
    EC_components_list=['EC_DT', 'EC_BLR', 'EC_Disk'],
    R_H=1.5E18,                  # distance of emission region from BH (cm)
    disk_type='MultiBB',
    silent=True
)

# Override central engine parameters with physically refined values
# These come from fitting to Sánchez Zaballa (2025) constraints + our Minuit results
prefit_jet.set_par('L_Disk',   val=1.3817e46)  # erg/s — disk luminosity
prefit_jet.set_par('M_BH',     val=9.8677e8)   # solar masses — black hole mass
prefit_jet.set_par('accr_eff', val=0.09)        # radiative efficiency η
prefit_jet.set_par('T_DT',     val=881.9)       # K — dust torus temperature
prefit_jet.set_par('tau_DT',   val=0.1)         # dust torus optical depth
prefit_jet.set_par('tau_BLR',  val=0.1)         # BLR optical depth

# Evaluate the model and plot against the data
prefit_jet.eval()
p = prefit_jet.plot_model(sed_data=sed_data)
p.add_residual_plot(prefit_jet, sed_data)
p.setlim(y_min=1E-16, x_min=1E8)
plt.suptitle('MG3 J163554+3629 — Physical prefit')
plt.show()

print('>>> Prefit complete.')
print(f'    Beaming factor δ : {prefit_jet.get_beaming():.2f}')
print(f'    B field          : {prefit_jet.parameters.B.val:.4f} G')
print(f'    R (emission region): {prefit_jet.parameters.R.val:.2e} cm')

---
## Step 6 — Add geometry and EBL absorption

### Opening angle geometry
We link the emission region size R to the jet geometry using the **opening angle** θ_open:

R = R_H × tan(θ_open)

This makes R a dependent parameter — it changes automatically whenever R_H changes during fitting, keeping the geometry physically consistent.

### EBL absorption
At z = 3.648, gamma-ray photons are **absorbed by the Extragalactic Background Light (EBL)** — the accumulated UV/optical/IR photons emitted by all galaxies since the Big Bang. This absorption is significant above ~10²³ Hz and must be included when modelling the gamma-ray SED.

We use the **Franceschini et al. (2008)** EBL model, implemented as a 2D template in JetSeT. The redshift of the EBL template is **linked** to the jet redshift so they are always consistent.

In [ ]:
# --- Geometry: link R to R_H via opening angle ---
# Add opening angle as a user-defined parameter
prefit_jet.add_user_par(name='theta_open', val=3, units='deg')

# Make R a dependent parameter: R = R_H * tan(theta_open)
# Important: must convert degrees to radians manually inside the expression
prefit_jet.make_dependent_par(
    par='R',
    depends_on=['R_H', 'theta_open'],
    par_expr='np.tan(np.radians(theta_open)) * R_H'
)

# Adjust R_H so that R stays at its pre-existing value
target_R = prefit_jet.parameters.R.val
new_RH   = target_R / np.tan(3 * np.pi / 180)
prefit_jet.set_par('R_H', val=new_RH)
prefit_jet.eval()

print(f'R_H set to: {prefit_jet.parameters.R_H.val:.3e} cm')
print(f'R (emission region): {prefit_jet.parameters.R.val:.3e} cm  (should match original)')

# --- EBL absorption ---
ebl = EBLAbsorptionTemplate.from_name('Franceschini_2008')

# Build composite fit model: jet + EBL
fit_model = FitModel(jet=prefit_jet, name='Final_Fit')
fit_model.add_component(ebl)

# Link EBL redshift to jet redshift — they must always be equal
fit_model.link_par(
    par_name='z_cosm',
    from_model='Franceschini_2008',
    to_model='jet_leptonic'
)

# Define composite expression: observed flux = jet emission × EBL transmission
fit_model.composite_expr = 'jet_leptonic * Franceschini_2008'

print(f'\n>>> Full model assembled at z = {prefit_jet.parameters.z_cosm.val}')
fit_model.show_model()

# Final evaluation and plot
fit_model.eval()
p = fit_model.plot_model(sed_data=sed_data)
p.add_residual_plot(fit_model, sed_data)
p.setlim(y_min=1E-16, x_min=1E8)
plt.suptitle('MG3 J163554+3629 — Full prefit model with EBL')
plt.show()

print('\n>>> Pipeline complete. fit_model is ready to pass to Minuit / MCMC / UltraNest.')

---
## Summary

After running this notebook you have:

- `sed_data` — the processed ObsData object (with systematics + ULs flagged)
- `sed_data_binned` — same, but rebinned at bin_width=0.15
- `my_shape` — SEDShape object with sync and IC peak characterisation
- `fit_model` — the full composite model (jet + EBL), ready for statistical fitting

These objects feed directly into the analysis notebooks in `analysis/one_zone/` and `analysis/two_zone/`.

---
*To adapt this pipeline for a different blazar: update the SED file path, the metadata (z, obj_name), the UL frequency list, the central engine parameters, and the ObsConstrain geometry arguments.*